In [ ]:
!pip install -q --upgrade unsloth transformers datasets trl accelerate bitsandbytes


In [ ]:
!nvidia-smi


Wed Oct  7 07:36:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.1: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.10.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "medalpaca/medical_meadow_medical_flashcards",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…): reconstructing file:   0%|          |  0.00B / 17.7MB            

medical_meadow_wikidoc_medical_flashcard(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 33955
})


In [ ]:
print(dataset[0])

{'input': 'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?', 'output': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.', 'instruction': 'Answer this question truthfully'}


In [ ]:
def format_example(example):
    return {
        "text": f"""### Question:
{example["input"]}

### Answer:
{example["output"]}"""
    }

dataset = dataset.map(format_example)

print(dataset[0]["text"])

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

### Question:
What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?

### Answer:
Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.


In [ ]:
def formatting_func(example):
    return tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": example["input"]
            },
            {
                "role": "assistant",
                "content": example["output"]
            }
        ],
        tokenize=False
    )

dataset = dataset.map(
    lambda x: {"text": formatting_func(x)}
)

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

In [ ]:
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|eot_id|>


In [ ]:
dataset = dataset.select(range(1000))

In [ ]:
print("Training examples:", len(dataset))

Training examples: 1000


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        output_dir="medical-llama-qlora",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
        optim="paged_adamw_8bit",
        report_to="none",
    ),
)

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
import torch

if torch.cuda.is_available():
    print(
        f"GPU memory allocated: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

    print(
        f"GPU memory reserved: "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
    )

GPU memory allocated: 2.23 GB
GPU memory reserved: 2.26 GB


In [ ]:
!nvidia-smi

Wed Oct  7 07:45:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             28W /   70W |    2465MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.728276
20,0.971994
30,0.894710
40,0.854962
50,0.897642
60,0.766354
70,0.710410
80,0.695986
90,0.651754
100,0.643571


Unsloth: Restored added_tokens_decoder metadata in medical-llama-qlora/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-llama-qlora/checkpoint-250/tokenizer_config.json.


In [ ]:
print(trainer_stats)

TrainOutput(global_step=250, training_loss=0.6415165176391602, metrics={'train_runtime': 578.4273, 'train_samples_per_second': 3.458, 'train_steps_per_second': 0.432, 'total_flos': 4741305009819648.0, 'train_loss': 0.6415165176391602, 'epoch': 2.0})


In [ ]:
model.save_pretrained("medical-llama-qlora")
tokenizer.save_pretrained("medical-llama-qlora")

Unsloth: Restored added_tokens_decoder metadata in medical-llama-qlora/tokenizer_config.json.


('medical-llama-qlora/tokenizer_config.json',
 'medical-llama-qlora/chat_template.jinja',
 'medical-llama-qlora/tokenizer.json')

In [ ]:
!ls -lh medical-llama-qlora

total 110M
-rw-r--r-- 1 root root 1.3K Oct  7 07:56 adapter_config.json
-rw------- 1 root root  93M Oct  7 07:56 adapter_model.safetensors
-rw-r--r-- 1 root root 3.8K Oct  7 07:56 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Oct  7 07:50 checkpoint-125
drwxr-xr-x 2 root root 4.0K Oct  7 07:55 checkpoint-250
-rw-r--r-- 1 root root 1.6K Oct  7 07:56 README.md
-rw-r--r-- 1 root root  50K Oct  7 07:56 tokenizer_config.json
-rw-r--r-- 1 root root  17M Oct  7 07:56 tokenizer.json


In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
question = "What are common symptoms of type 2 diabetes?"

messages = [
    {
        "role": "user",
        "content": question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What are common symptoms of type 2 diabetes?assistant

Type 2 diabetes is often associated with obesity and a sedentary lifestyle, which can cause fatigue and weight gain. Other symptoms may include increased thirst and urination, and darkened skin in some cases.


In [ ]:
questions = [
    "What are common symptoms of type 2 diabetes?",
    "What are the common symptoms of anemia?",
    "What causes high blood pressure?",
    "What are common symptoms of dehydration?",
    "What is the difference between a viral and bacterial infection?"
]

for question in questions:

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 80)
    print("QUESTION:", question)
    print("RESPONSE:", response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What are common symptoms of type 2 diabetes?
RESPONSE: system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What are common symptoms of type 2 diabetes?assistant

Type 2 diabetes is often asymptomatic, but it may present with polyuria, polydipsia, and unexplained weight loss.


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What are the common symptoms of anemia?
RESPONSE: system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What are the common symptoms of anemia?assistant

Anemia is a condition characterized by a decrease in the number of red blood cells or the amount of hemoglobin in the blood. The common symptoms of anemia include fatigue, weakness, and shortness of breath. These symptoms can vary in severity and may be accompanied by other symptoms such as pallor, cold hands and feet, and dizziness. In some cases, anemia may also cause difficulty concentrating and memory problems. It is important to seek medical attention if you experience any of these symptoms, as anemia can be treated with medication or other interventions.


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What causes high blood pressure?
RESPONSE: system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What causes high blood pressure?assistant

High blood pressure is caused by a variety of factors, including aortic stenosis, endocrine disorders such as Cushing's syndrome, renal disease, and certain medications.


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What are common symptoms of dehydration?
RESPONSE: system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What are common symptoms of dehydration?assistant

Dehydration is characterized by decreased urine output and dry mouth.
QUESTION: What is the difference between a viral and bacterial infection?
RESPONSE: system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What is the difference between a viral and bacterial infection?assistant

Viral and bacterial infections can be differentiated by their effects on the body's white blood cells. Viral infections typically cause a decrease in white blood cell count, while bacterial infections cause an increase in white blood cell count.


In [ ]:
question = "What are common symptoms of anemia?"

messages = [
    {
        "role": "user",
        "content": question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", question)
print("Model Response:")
print(response)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are common symptoms of anemia?
Model Response:
system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What are common symptoms of anemia?assistant

Anemia often presents with fatigue, weakness, and shortness of breath.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!cp -r medical-llama-qlora /content/drive/MyDrive/

In [ ]:
from unsloth import FastLanguageModel

# Prepare model for inference
FastLanguageModel.for_inference(model)

print("Model is ready for testing!")

Model is ready for testing!


In [ ]:
question = "What are common symptoms of anemia?"

messages = [
    {
        "role": "user",
        "content": question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", question)
print("\nModel Response:")
print(response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are common symptoms of anemia?

Model Response:
system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What are common symptoms of anemia?assistant

Anemia is often characterized by fatigue and weakness.


In [ ]:
question = "What causes dehydration?"

messages = [
    {
        "role": "user",
        "content": question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", question)
print("\nModel Response:")
print(response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What causes dehydration?

Model Response:
system

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2026

user

What causes dehydration?assistant

Dehydration is a condition that can result from the loss of bodily fluids. This can occur due to various factors, including excessive sweating, diarrhea, vomiting, fever, and blood loss. Additionally, certain medications and medical conditions can also contribute to the development of dehydration. It is important to recognize the potential causes of dehydration in order to provide appropriate treatment and management of the condition.


In [ ]:
print("Training completed successfully.")
print("Final training loss:", trainer_stats.training_loss)

Training completed successfully.
Final training loss: 0.6415165176391602


In [ ]:
import torch

print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 2.45 GB
GPU memory reserved: 2.47 GB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls -lh /content/drive/MyDrive/medical-llama-qlora

total 110M
-rw------- 1 root root 1.3K Oct  7 08:01 adapter_config.json
-rw------- 1 root root  93M Oct  7 08:01 adapter_model.safetensors
-rw------- 1 root root 3.8K Oct  7 08:01 chat_template.jinja
drwx------ 2 root root 4.0K Oct  7 08:01 checkpoint-125
drwx------ 2 root root 4.0K Oct  7 08:01 checkpoint-250
-rw------- 1 root root 1.6K Oct  7 08:01 README.md
-rw------- 1 root root  50K Oct  7 08:01 tokenizer_config.json
-rw------- 1 root root  17M Oct  7 08:01 tokenizer.json


In [38]:
# Download your Colab notebook
from google.colab import files
files.download('/content/medical_qlora.ipynb')

FileNotFoundError: Cannot find file: /content/medical_qlora.ipynb

In [39]:
from google.colab import files
from google.colab import drive

# Save current Colab notebook to Google Drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/Medical-QLoRA-Unsloth"

!cp -r "/content/medical-llama-qlora" "/content/drive/MyDrive/Medical-QLoRA-Unsloth/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
